In [ ]:
  ### Cell 1 — locate the dataset and load its tables

  from pathlib import Path
  import random

  import numpy as np
  import pandas as pd
  import torch

  SEED = 42

  random.seed(SEED)
  np.random.seed(SEED)
  torch.manual_seed(SEED)
  if torch.cuda.is_available():
      torch.cuda.manual_seed_all(SEED)

  # Kaggle may expose competition data in either location.
  data_candidates = [
      Path("/kaggle/input/rsna-knee-abnormality-detection"),
      Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
  ]

  DATA = next((path for path in data_candidates if path.is_dir()), None)

  if DATA is None:
      raise FileNotFoundError(
          "Competition data was not found. Attach "
          "'rsna-knee-abnormality-detection' to this notebook."
      )

  csv_paths = sorted(DATA.glob("*.csv"))

  print("Dataset directory:", DATA)
  print("CSV files:", [path.name for path in csv_paths])
  print("CUDA available:", torch.cuda.is_available())
  if torch.cuda.is_available():
      print("GPU:", torch.cuda.get_device_name(0))

  train = pd.read_csv(DATA / "train.csv")
  train_series = pd.read_csv(DATA / "train_series.csv")
  test = pd.read_csv(DATA / "test.csv")
  test_series = pd.read_csv(DATA / "test_series.csv")
  sample_submission = pd.read_csv(DATA / "sample_submission.csv")

  print()
  print("train:", train.shape)
  print("train_series:", train_series.shape)
  print("test:", test.shape)
  print("test_series:", test_series.shape)
  print("sample_submission:", sample_submission.shape)

  display(train.head(3))
  display(train_series.head(3))

In [ ]:
  ### Cell 2 — audit the 12 gold labels

  TARGETS = [
      column
      for column in sample_submission.columns
      if column != "StudyInstanceUID"
  ]

  print("Number of targets:", len(TARGETS))
  print("Targets:")
  for target in TARGETS:
      print(" -", target)

  label_mask = train[TARGETS].notna()

  studies_with_any_label = label_mask.any(axis=1)
  studies_with_all_labels = label_mask.all(axis=1)

  print()
  print("Training studies:", train["StudyInstanceUID"].nunique())
  print("Studies with at least one gold label:", studies_with_any_label.sum())
  print("Studies with all 12 gold labels:", studies_with_all_labels.sum())
  print("Studies without gold labels:", (~studies_with_any_label).sum())
  print("Missing reports:", train["Report"].isna().sum())
  print("Duplicate study rows:", train["StudyInstanceUID"].duplicated().sum())

  label_summary = pd.DataFrame(
      {
          "known": label_mask.sum(),
          "missing": label_mask.isna().sum(),  # corrected below
          "positive": train[TARGETS].eq(1).sum(),
          "negative": train[TARGETS].eq(0).sum(),
      }
  )

  # `label_mask` is Boolean, so count unknown labels directly from the source table.
  label_summary["missing"] = train[TARGETS].isna().sum()
  label_summary["positive_rate"] = (
      label_summary["positive"] / label_summary["known"]
  )

  display(label_summary)

  observed_values = {
      target: sorted(train[target].dropna().unique().tolist())
      for target in TARGETS
  }

  print("Observed values per target:")
  for target, values in observed_values.items():
      print(f"{target}: {values}")


In [ ]:
  ### Cell 3 — audit the study/series structure and discover the image hierarchy

  # Work with string IDs consistently. DICOM UIDs are identifiers, not numbers.
  train["StudyInstanceUID"] = train["StudyInstanceUID"].astype(str)
  train_series["StudyInstanceUID"] = train_series["StudyInstanceUID"].astype(str)
  train_series["SeriesInstanceUID"] = train_series["SeriesInstanceUID"].astype(str)

  labeled_study_ids = set(
      train.loc[studies_with_all_labels, "StudyInstanceUID"]
  )

  print("Training series rows:", len(train_series))
  print("Unique series IDs:", train_series["SeriesInstanceUID"].nunique())
  print(
      "Duplicate series IDs:",
      train_series["SeriesInstanceUID"].duplicated().sum(),
  )

  series_without_study = ~train_series["StudyInstanceUID"].isin(
      train["StudyInstanceUID"]
  )

  print(
      "Series whose study is absent from train.csv:",
      series_without_study.sum(),
  )

  series_counts = (
      train_series.groupby("StudyInstanceUID")
      .size()
      .rename("series_count")
  )

  print()
  print("Series per training study:")
  display(
      series_counts.describe(
          percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
      ).to_frame()
  )

  labeled_series_counts = series_counts[
      series_counts.index.isin(labeled_study_ids)
  ]

  print("Series per gold-labeled study:")
  display(
      labeled_series_counts.describe(
          percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
      ).to_frame()
  )

  print("Series metadata missing values:")
  display(
      train_series[
          ["Fluid_Sensitive", "Fat_Suppression", "Anatomical_Plane"]
      ]
      .isna()
      .sum()
      .to_frame("missing")
  )

  print("Series protocol combinations:")
  display(
      train_series[
          ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]
      ]
      .value_counts()
      .rename("series")
      .to_frame()
  )

  # Locate one DICOM file only. This does not decode its pixels.
  first_dicom = next(DATA.rglob("*.dcm"))

  print()
  print("Example DICOM path relative to dataset root:")
  print(first_dicom.relative_to(DATA))

  print()
  print("Its directory hierarchy:")
  current = first_dicom.parent
  while current != DATA:
      print(" -", current.relative_to(DATA))
      current = current.parent


In [ ]:
### Cell 4 — audit DICOM geometry for one labeled study

%pip install -q pydicom==3.0.1



In [ ]:

import pydicom

TRAIN_DICOM_ROOT = DATA / "train_series"

assert TRAIN_DICOM_ROOT.is_dir(), TRAIN_DICOM_ROOT


def plane_from_normal(normal, threshold=0.95):
  """
  Infer the closest anatomical plane from a DICOM LPS slice normal.

  LPS axes:
    x -> left/right
    y -> posterior/anterior
    z -> inferior/superior
  """
  dominant_axis = int(np.argmax(np.abs(normal)))
  dominant_value = float(np.max(np.abs(normal)))

  if dominant_value < threshold:
      return "Oblique"

  return {
      0: "Sagittal",
      1: "Coronal",
      2: "Axial",
  }[dominant_axis]


def instance_number_direction(instance_numbers):
  values = np.asarray(instance_numbers, dtype=float)
  differences = np.diff(values)

  if np.all(differences > 0):
      return "increasing"
  if np.all(differences < 0):
      return "decreasing"
  return "mixed"


# Choose the first gold-labeled study deterministically.
example_study_uid = sorted(labeled_study_ids)[0]

example_series = (
  train_series[
      train_series["StudyInstanceUID"] == example_study_uid
  ]
  .sort_values(["Anatomical_Plane", "SeriesInstanceUID"])
  .reset_index(drop=True)
)

print("Example labeled study")
print("Number of series:", len(example_series))
print()

geometry_rows = []
patient_ids = set()

for series_row in example_series.itertuples(index=False):
  folder = (
      TRAIN_DICOM_ROOT
      / str(series_row.StudyInstanceUID)
      / str(series_row.SeriesInstanceUID)
  )

  if not folder.is_dir():
      raise FileNotFoundError(folder)

  dicom_paths = sorted(folder.glob("*.dcm"))

  if len(dicom_paths) < 2:
      raise ValueError(
          f"Expected at least two slices in {folder}, "
          f"found {len(dicom_paths)}"
      )

  # Read headers only. Pixel arrays are not decoded in this cell.
  headers = [
      pydicom.dcmread(path, stop_before_pixels=True)
      for path in dicom_paths
  ]

  study_uids = {
      str(header.StudyInstanceUID)
      for header in headers
  }
  series_uids = {
      str(header.SeriesInstanceUID)
      for header in headers
  }
  sop_uids = [
      str(header.SOPInstanceUID)
      for header in headers
  ]

  if len(study_uids) != 1:
      raise ValueError("Mixed StudyInstanceUID values in one directory")

  if len(series_uids) != 1:
      raise ValueError("Mixed SeriesInstanceUID values in one directory")

  if len(set(sop_uids)) != len(sop_uids):
      raise ValueError("Duplicate SOPInstanceUID values")

  for header in headers:
      if hasattr(header, "PatientID"):
          patient_ids.add(str(header.PatientID))

  required_geometry = [
      "ImagePositionPatient",
      "ImageOrientationPatient",
      "PixelSpacing",
      "Rows",
      "Columns",
  ]

  missing_geometry = {
      field: sum(
          not hasattr(header, field)
          for header in headers
      )
      for field in required_geometry
  }

  if any(missing_geometry.values()):
      raise ValueError(
          f"Missing geometry in series "
          f"{series_row.SeriesInstanceUID}: {missing_geometry}"
      )

  reference_orientation = np.asarray(
      headers[0].ImageOrientationPatient,
      dtype=np.float64,
  )

  orientations = np.asarray(
      [
          header.ImageOrientationPatient
          for header in headers
      ],
      dtype=np.float64,
  )

  if not np.allclose(
      orientations,
      reference_orientation,
      atol=1e-4,
      rtol=0,
  ):
      raise ValueError("Orientation varies within series")

  row_direction = reference_orientation[:3]
  column_direction = reference_orientation[3:]
  slice_normal = np.cross(
      row_direction,
      column_direction,
  )

  positions = np.asarray(
      [
          header.ImagePositionPatient
          for header in headers
      ],
      dtype=np.float64,
  )

  # Physical coordinate of each slice along the slice normal.
  projections = positions @ slice_normal
  physical_order = np.argsort(projections, kind="stable")
  ordered_projections = projections[physical_order]
  slice_gaps = np.diff(ordered_projections)

  if np.any(slice_gaps <= 1e-3):
      raise ValueError(
          "Repeated or indistinguishable physical slice positions"
      )

  median_gap = float(np.median(slice_gaps))
  maximum_gap_error = float(
      np.max(np.abs(slice_gaps - median_gap))
  )

  ordered_instance_numbers = [
      int(getattr(headers[index], "InstanceNumber", -1))
      for index in physical_order
  ]

  pixel_spacing = np.asarray(
      headers[0].PixelSpacing,
      dtype=np.float64,
  )

  derived_plane = plane_from_normal(slice_normal)

  geometry_rows.append(
      {
          "supplied_plane": series_row.Anatomical_Plane,
          "derived_plane": derived_plane,
          "fluid_sensitive": series_row.Fluid_Sensitive,
          "fat_suppression": series_row.Fat_Suppression,
          "slices": len(headers),
          "rows": int(headers[0].Rows),
          "columns": int(headers[0].Columns),
          "row_spacing_mm": pixel_spacing[0],
          "column_spacing_mm": pixel_spacing[1],
          "physical_slice_gap_mm": median_gap,
          "max_gap_error_mm": maximum_gap_error,
          "slice_thickness_mm": getattr(
              headers[0],
              "SliceThickness",
              np.nan,
          ),
          "instance_numbers_in_physical_order":
              instance_number_direction(
                  ordered_instance_numbers
              ),
          "series_description": getattr(
              headers[0],
              "SeriesDescription",
              "",
          ),
          "protocol_name": getattr(
              headers[0],
              "ProtocolName",
              "",
          ),
          "manufacturer": getattr(
              headers[0],
              "Manufacturer",
              "",
          ),
          "model": getattr(
              headers[0],
              "ManufacturerModelName",
              "",
          ),
      }
  )

geometry_audit = pd.DataFrame(geometry_rows)

print("Distinct PatientID values in this study:", len(patient_ids))
display(geometry_audit)

In [ ]:
### Cell 5 — reconstruct and visualize one real volume

import matplotlib.pyplot as plt


def reconstruct_series(study_uid, series_uid):
  """
  Reconstruct one classic single-frame MRI series.

  Returns
  -------
  volume:
      Float32 NumPy array with axes (slice, row, column).
  geometry:
      Spatial metadata describing the reconstructed array.
  """
  folder = (
      TRAIN_DICOM_ROOT
      / str(study_uid)
      / str(series_uid)
  )

  dicom_paths = sorted(folder.glob("*.dcm"))

  if len(dicom_paths) < 2:
      raise ValueError(
          f"Expected at least two DICOM slices: {folder}"
      )

  headers = [
      pydicom.dcmread(path, stop_before_pixels=True)
      for path in dicom_paths
  ]

  reference_orientation = np.asarray(
      headers[0].ImageOrientationPatient,
      dtype=np.float64,
  )

  orientations = np.asarray(
      [
          header.ImageOrientationPatient
          for header in headers
      ],
      dtype=np.float64,
  )

  if not np.allclose(
      orientations,
      reference_orientation,
      atol=1e-4,
      rtol=0,
  ):
      raise ValueError("Orientation varies within the series")

  pixel_spacing = np.asarray(
      headers[0].PixelSpacing,
      dtype=np.float64,
  )

  for header in headers:
      current_spacing = np.asarray(
          header.PixelSpacing,
          dtype=np.float64,
      )

      if not np.allclose(
          current_spacing,
          pixel_spacing,
          atol=1e-4,
          rtol=0,
      ):
          raise ValueError(
              "PixelSpacing varies within the series"
          )

  # DICOM IOP stores the directions across columns and rows.
  column_axis_direction = reference_orientation[:3]
  row_axis_direction = reference_orientation[3:]

  slice_normal = np.cross(
      column_axis_direction,
      row_axis_direction,
  )

  positions = np.asarray(
      [
          header.ImagePositionPatient
          for header in headers
      ],
      dtype=np.float64,
  )

  projections = positions @ slice_normal
  physical_order = np.argsort(
      projections,
      kind="stable",
  )

  ordered_positions = positions[physical_order]
  ordered_projections = projections[physical_order]
  slice_gaps = np.diff(ordered_projections)

  if np.any(slice_gaps <= 1e-3):
      raise ValueError(
          "Repeated or indistinguishable slice positions"
      )

  slice_spacing = float(np.median(slice_gaps))

  if not np.allclose(
      slice_gaps,
      slice_spacing,
      atol=0.05,
      rtol=0.01,
  ):
      raise ValueError(
          "Irregular spacing or missing slices"
      )

  ordered_slices = []
  ordered_files = []

  for original_index in physical_order:
      path = dicom_paths[original_index]
      dcm = pydicom.dcmread(path)

      pixels = dcm.pixel_array.astype(np.float32)

      if pixels.shape != (
          int(dcm.Rows),
          int(dcm.Columns),
      ):
          raise ValueError(
              f"Unexpected pixel shape: {pixels.shape}"
          )

      # Exclude declared padding from intensity statistics.
      if hasattr(dcm, "PixelPaddingValue"):
          padding_value = float(dcm.PixelPaddingValue)

          if hasattr(dcm, "PixelPaddingRangeLimit"):
              padding_limit = float(
                  dcm.PixelPaddingRangeLimit
              )
          else:
              padding_limit = padding_value

          padding_low = min(
              padding_value,
              padding_limit,
          )
          padding_high = max(
              padding_value,
              padding_limit,
          )

          padding_mask = (
              (pixels >= padding_low)
              & (pixels <= padding_high)
          )
      else:
          padding_mask = np.zeros(
              pixels.shape,
              dtype=bool,
          )

      slope = float(
          getattr(dcm, "RescaleSlope", 1.0)
      )
      intercept = float(
          getattr(dcm, "RescaleIntercept", 0.0)
      )

      pixels = pixels * slope + intercept
      pixels[padding_mask] = np.nan

      if not np.isfinite(pixels).any():
          raise ValueError(
              f"No finite pixels in {path.name}"
          )

      ordered_slices.append(pixels)
      ordered_files.append(str(path))

  volume = np.stack(
      ordered_slices,
      axis=0,
  ).astype(np.float32)

  # Array axes are (slice, row, column).
  # This affine maps those indices into DICOM LPS millimetres.
  affine = np.eye(4, dtype=np.float64)
  affine[:3, 0] = (
      slice_normal * slice_spacing
  )
  affine[:3, 1] = (
      row_axis_direction * pixel_spacing[0]
  )
  affine[:3, 2] = (
      column_axis_direction * pixel_spacing[1]
  )
  affine[:3, 3] = ordered_positions[0]

  geometry = {
      "array_axes": ("slice", "row", "column"),
      "coordinate_system": "DICOM LPS millimetres",
      "spacing_drc_mm": (
          slice_spacing,
          float(pixel_spacing[0]),
          float(pixel_spacing[1]),
      ),
      "affine_array_to_lps": affine,
      "slice_normal_lps": slice_normal,
      "ordered_positions_lps": ordered_positions,
      "ordered_files": ordered_files,
      "photometric_interpretation": str(
          getattr(
              headers[0],
              "PhotometricInterpretation",
              "",
          )
      ),
  }

  return volume, geometry


# Select the fluid-sensitive sagittal series from the audited study.
chosen_row = example_series[
  (example_series["Anatomical_Plane"] == "Sagittal")
  & (example_series["Fluid_Sensitive"] == 1)
].iloc[0]

volume, geometry = reconstruct_series(
  chosen_row["StudyInstanceUID"],
  chosen_row["SeriesInstanceUID"],
)

slice_spacing, row_spacing, column_spacing = (
  geometry["spacing_drc_mm"]
)

finite_pixels = volume[np.isfinite(volume)]
display_low, display_high = np.percentile(
  finite_pixels,
  [1, 99],
)

print("Volume shape (slice, row, column):", volume.shape)
print("Voxel spacing in mm:", geometry["spacing_drc_mm"])
print(
  "Physical field of view in mm:",
  tuple(
      round(size * spacing, 2)
      for size, spacing in zip(
          volume.shape,
          geometry["spacing_drc_mm"],
      )
  ),
)
print(
  "Finite intensity range:",
  float(finite_pixels.min()),
  float(finite_pixels.max()),
)
print(
  "Display window (1st–99th percentile):",
  float(display_low),
  float(display_high),
)
print(
  "Photometric interpretation:",
  geometry["photometric_interpretation"],
)

fig, axes = plt.subplots(
  2,
  3,
  figsize=(13, 8),
)

# Three slices in the native acquisition plane.
slice_indices = np.linspace(
  0,
  volume.shape[0] - 1,
  3,
).astype(int)

for axis, slice_index in zip(
  axes[0],
  slice_indices,
):
  axis.imshow(
      volume[slice_index],
      cmap="gray",
      vmin=display_low,
      vmax=display_high,
      aspect=row_spacing / column_spacing,
  )
  axis.set_title(
      f"Native slice {slice_index}"
  )
  axis.set_xlabel("column")
  axis.set_ylabel("row")

# Orthogonal views through the centre of the native volume.
middle_row = volume.shape[1] // 2
middle_column = volume.shape[2] // 2

axes[1, 0].imshow(
  volume[:, middle_row, :],
  cmap="gray",
  vmin=display_low,
  vmax=display_high,
  aspect=slice_spacing / column_spacing,
)
axes[1, 0].set_title("Centre row")
axes[1, 0].set_xlabel("column")
axes[1, 0].set_ylabel("physically ordered slice")

axes[1, 1].imshow(
  volume[:, :, middle_column],
  cmap="gray",
  vmin=display_low,
  vmax=display_high,
  aspect=slice_spacing / row_spacing,
)
axes[1, 1].set_title("Centre column")
axes[1, 1].set_xlabel("row")
axes[1, 1].set_ylabel("physically ordered slice")

axes[1, 2].axis("off")
axes[1, 2].text(
  0,
  0.8,
  "Display only:\n"
  "• physical slice ordering\n"
  "• physical aspect ratios\n"
  "• 1st–99th percentile window\n\n"
  "No resampling\n"
  "No cropping\n"
  "No model normalization",
  va="top",
  fontsize=11,
)

plt.tight_layout()
plt.show()

In [ ]:
### Cell 6 — compare sagittal, coronal, and axial reconstructions

# Prefer a fluid-sensitive series for each anatomical plane.
representative_rows = (
  example_series
  .sort_values(
      ["Anatomical_Plane", "Fluid_Sensitive"],
      ascending=[True, False],
  )
  .groupby("Anatomical_Plane", as_index=False)
  .head(1)
  .sort_values("Anatomical_Plane")
  .reset_index(drop=True)
)

fig, axes = plt.subplots(
  len(representative_rows),
  3,
  figsize=(13, 4 * len(representative_rows)),
)

reconstruction_summary = []

for row_index, series_row in representative_rows.iterrows():
  plane = series_row["Anatomical_Plane"]

  plane_volume, plane_geometry = reconstruct_series(
      series_row["StudyInstanceUID"],
      series_row["SeriesInstanceUID"],
  )

  (
      plane_slice_spacing,
      plane_row_spacing,
      plane_column_spacing,
  ) = plane_geometry["spacing_drc_mm"]

  finite = plane_volume[
      np.isfinite(plane_volume)
  ]

  window_low, window_high = np.percentile(
      finite,
      [1, 99],
  )

  middle_slice = plane_volume.shape[0] // 2
  middle_row = plane_volume.shape[1] // 2
  middle_column = plane_volume.shape[2] // 2

  # Native acquisition plane.
  axes[row_index, 0].imshow(
      plane_volume[middle_slice],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_row_spacing
          / plane_column_spacing
      ),
  )
  axes[row_index, 0].set_title(
      f"{plane}: native centre slice"
  )
  axes[row_index, 0].set_xlabel("column")
  axes[row_index, 0].set_ylabel("row")

  # Orthogonal centre-row view.
  axes[row_index, 1].imshow(
      plane_volume[:, middle_row, :],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_slice_spacing
          / plane_column_spacing
      ),
  )
  axes[row_index, 1].set_title(
      f"{plane}: centre row"
  )
  axes[row_index, 1].set_xlabel("column")
  axes[row_index, 1].set_ylabel(
      "physically ordered slice"
  )

  # Orthogonal centre-column view.
  axes[row_index, 2].imshow(
      plane_volume[:, :, middle_column],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_slice_spacing
          / plane_row_spacing
      ),
  )
  axes[row_index, 2].set_title(
      f"{plane}: centre column"
  )
  axes[row_index, 2].set_xlabel("row")
  axes[row_index, 2].set_ylabel(
      "physically ordered slice"
  )

  reconstruction_summary.append(
      {
          "plane": plane,
          "fluid_sensitive":
              series_row["Fluid_Sensitive"],
          "fat_suppression":
              series_row["Fat_Suppression"],
          "shape_drc": tuple(
              plane_volume.shape
          ),
          "spacing_drc_mm":
              plane_geometry["spacing_drc_mm"],
          "physical_fov_mm": tuple(
              round(size * spacing, 2)
              for size, spacing in zip(
                  plane_volume.shape,
                  plane_geometry[
                      "spacing_drc_mm"
                  ],
              )
          ),
      }
  )

plt.tight_layout()
plt.show()

display(
  pd.DataFrame(reconstruction_summary)
)

In [ ]:
### Cell 7 — audit patient identifiers and scanner metadata


first_series_per_study = (
  train_series
  .sort_values(
      ["StudyInstanceUID", "SeriesInstanceUID"]
  )
  .drop_duplicates(
      subset="StudyInstanceUID",
      keep="first",
  )
)

study_metadata_rows = []

total_studies = len(first_series_per_study)

for index, series_row in enumerate(
  first_series_per_study.itertuples(index=False),
  start=1,
):
  folder = (
      TRAIN_DICOM_ROOT
      / str(series_row.StudyInstanceUID)
      / str(series_row.SeriesInstanceUID)
  )

  first_path = next(
      folder.glob("*.dcm"),
      None,
  )

  if first_path is None:
      raise FileNotFoundError(
          f"No DICOM files found in {folder}"
      )

  header = pydicom.dcmread(
      first_path,
      stop_before_pixels=True,
      specific_tags=[
          "PatientID",
          "StudyInstanceUID",
          "Manufacturer",
          "ManufacturerModelName",
          "MagneticFieldStrength",
          "InstitutionName",
          "StationName",
      ],
  )

  header_study_uid = str(
      getattr(
          header,
          "StudyInstanceUID",
          "",
      )
  )

  if header_study_uid != str(
      series_row.StudyInstanceUID
  ):
      raise ValueError(
          "CSV and DICOM StudyInstanceUID disagree"
      )

  study_metadata_rows.append(
      {
          "StudyInstanceUID":
              header_study_uid,
          "PatientID": str(
              getattr(header, "PatientID", "")
          ),
          "Manufacturer": str(
              getattr(header, "Manufacturer", "")
          ),
          "ManufacturerModelName": str(
              getattr(
                  header,
                  "ManufacturerModelName",
                  "",
              )
          ),
          "MagneticFieldStrength": str(
              getattr(
                  header,
                  "MagneticFieldStrength",
                  "",
              )
          ),
          "InstitutionName": str(
              getattr(
                  header,
                  "InstitutionName",
                  "",
              )
          ),
          "StationName": str(
              getattr(header, "StationName", "")
          ),
      }
  )

  if (
      index % 500 == 0
      or index == total_studies
  ):
      print(
          f"Read {index}/{total_studies} "
          "study headers"
      )

study_metadata = pd.DataFrame(
  study_metadata_rows
)

study_metadata["has_gold_labels"] = (
  study_metadata["StudyInstanceUID"]
  .isin(labeled_study_ids)
)

missing_patient_id = (
  study_metadata["PatientID"]
  .str.strip()
  .eq("")
)

usable_patient_metadata = study_metadata[
  ~missing_patient_id
]

studies_per_patient = (
  usable_patient_metadata
  .groupby("PatientID")
  ["StudyInstanceUID"]
  .nunique()
)

print()
print("Studies audited:", len(study_metadata))
print(
  "Studies missing PatientID:",
  int(missing_patient_id.sum()),
)
print(
  "Unique non-empty PatientIDs:",
  usable_patient_metadata[
      "PatientID"
  ].nunique(),
)
print(
  "Patients with multiple studies:",
  int((studies_per_patient > 1).sum()),
)
print(
  "Maximum studies for one patient:",
  int(studies_per_patient.max()),
)

print()
print("Studies per patient:")
display(
  studies_per_patient
  .describe(
      percentiles=[
          0.5,
          0.9,
          0.95,
          0.99,
      ]
  )
  .to_frame("studies_per_patient")
)

print("Scanner distribution:")
scanner_distribution = (
  study_metadata
  .groupby(
      [
          "Manufacturer",
          "ManufacturerModelName",
          "MagneticFieldStrength",
      ],
      dropna=False,
  )
  .agg(
      studies=(
          "StudyInstanceUID",
          "nunique",
      ),
      gold_studies=(
          "has_gold_labels",
          "sum",
      ),
  )
  .sort_values(
      "studies",
      ascending=False,
  )
)

display(scanner_distribution)

print("Missing or empty scanner/site fields:")
display(
  pd.DataFrame(
      {
          column: {
              "empty": int(
                  study_metadata[column]
                  .str.strip()
                  .eq("")
                  .sum()
              ),
              "unique_non_empty": int(
                  study_metadata.loc[
                      ~study_metadata[column]
                      .str.strip()
                      .eq(""),
                      column,
                  ].nunique()
              ),
          }
          for column in [
              "Manufacturer",
              "ManufacturerModelName",
              "MagneticFieldStrength",
              "InstitutionName",
              "StationName",
          ]
      }
  ).T
)


In [ ]:
### Cell 8 — verify Pillar-0 checkpoint access

%pip install -q \
  "transformers==4.56.2" \
  "huggingface_hub>=0.34,<1.0" \
  "accelerate==1.10.1"


In [ ]:
import transformers
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

MODEL_ID = "YalaLab/Pillar0-BreastMRI"
MODEL_REVISION = "main"

try:
  hf_token = (
      UserSecretsClient()
      .get_secret("HF_TOKEN")
  )
except Exception as error:
  raise RuntimeError(
      "Kaggle secret HF_TOKEN is missing or "
      "not enabled for this notebook."
  ) from error

if not hf_token:
  raise RuntimeError(
      "Kaggle secret HF_TOKEN is empty."
  )

hf_api = HfApi(token=hf_token)

try:
  model_info = hf_api.model_info(
      MODEL_ID,
      revision=MODEL_REVISION,
      files_metadata=True,
  )
except Exception as error:
  raise RuntimeError(
      "The Hugging Face token could not access "
      f"{MODEL_ID}. Confirm that the account has "
      "accepted the gated model conditions."
  ) from error

model_files = pd.DataFrame(
  [
      {
          "filename": file.rfilename,
          "size_mb": (
              file.size / 1024**2
              if file.size is not None
              else np.nan
          ),
      }
      for file in model_info.siblings
  ]
).sort_values("filename")

print("Transformers version:", transformers.__version__)
print("Model:", model_info.id)
print("Resolved revision:", model_info.sha)
print("Gated:", model_info.gated)
print(
  "Total repository size (MB):",
  round(
      model_files["size_mb"].sum(
          skipna=True
      ),
      1,
  ),
)

display(model_files)


In [ ]:
import json

from transformers import AutoModel

PILLAR_REVISION = (
  "be5050ba482607f90e607e044f03a2ebc38f98c3"
)

if not torch.cuda.is_available():
  raise RuntimeError(
      "Enable a Kaggle GPU before loading Pillar-0."
  )

device = torch.device("cuda")

# Make the cell safe to rerun without loading a second model.
if "pillar" not in globals():
  torch.cuda.empty_cache()

  pillar = AutoModel.from_pretrained(
      MODEL_ID,
      revision=PILLAR_REVISION,
      token=hf_token,
      trust_remote_code=True,
      low_cpu_mem_usage=True,
  )

pillar = pillar.eval().to(device)

for parameter in pillar.parameters():
  parameter.requires_grad_(False)

parameter_count = sum(
  parameter.numel()
  for parameter in pillar.parameters()
)

trainable_parameter_count = sum(
  parameter.numel()
  for parameter in pillar.parameters()
  if parameter.requires_grad
)

first_parameter = next(
  pillar.parameters()
)

print("Model class:", type(pillar))
print("Parameters:", f"{parameter_count:,}")
print(
  "Trainable parameters:",
  trainable_parameter_count,
)
print("Training mode:", pillar.training)
print("Device:", first_parameter.device)
print("Parameter dtype:", first_parameter.dtype)
print(
  "GPU memory allocated (MB):",
  round(
      torch.cuda.memory_allocated()
      / 1024**2,
      1,
  ),
)
print(
  "GPU memory reserved (MB):",
  round(
      torch.cuda.memory_reserved()
      / 1024**2,
      1,
  ),
)

print()
print("Model configuration:")
print(
  json.dumps(
      pillar.config.to_dict(),
      indent=2,
      default=str,
  )
)



In [ ]:
### Cell 10 — canonicalize and preprocess one series

from itertools import product

from scipy.ndimage import affine_transform


TARGET_SPACING_XYZ_MM = np.array(
  [1.0, 1.0, 1.0],
  dtype=np.float64,
)

TARGET_SHAPE_ZYX = (
  192,
  384,
  384,
)


def canonical_resample_lps(
  native_volume,
  native_affine,
  target_spacing_xyz_mm=TARGET_SPACING_XYZ_MM,
):
  """Resample signal and validity together, avoiding padding bleed.

  Returns intensity normalized by interpolated support, a fractional support
  map in [0,1], and canonical-grid metadata. A genuine zero remains supported.
  """
  native_shape = np.asarray(native_volume.shape, dtype=int)
  native_validity = np.isfinite(native_volume).astype(np.float32)
  native_signal = np.where(
      native_validity > 0,
      np.nan_to_num(native_volume, nan=0.0, posinf=0.0, neginf=0.0),
      0.0,
  ).astype(np.float32)

  native_corners = np.asarray(list(product(
      [0, native_shape[0] - 1],
      [0, native_shape[1] - 1],
      [0, native_shape[2] - 1],
  )), dtype=np.float64)
  homogeneous_corners = np.concatenate(
      [native_corners, np.ones((len(native_corners), 1), dtype=np.float64)], axis=1
  )
  lps_corners = (native_affine @ homogeneous_corners.T).T[:, :3]
  lps_min = lps_corners.min(axis=0)
  lps_max = lps_corners.max(axis=0)
  physical_extent_xyz = lps_max - lps_min
  output_size_xyz = np.ceil(physical_extent_xyz / target_spacing_xyz_mm).astype(int) + 1
  output_shape_zyx = tuple(output_size_xyz[::-1])

  canonical_affine = np.eye(4, dtype=np.float64)
  canonical_affine[:3, 0] = [0, 0, target_spacing_xyz_mm[2]]
  canonical_affine[:3, 1] = [0, target_spacing_xyz_mm[1], 0]
  canonical_affine[:3, 2] = [target_spacing_xyz_mm[0], 0, 0]
  canonical_affine[:3, 3] = lps_min
  output_to_input = np.linalg.inv(native_affine) @ canonical_affine
  transform = dict(
      matrix=output_to_input[:3, :3],
      offset=output_to_input[:3, 3],
      output_shape=output_shape_zyx,
      order=1,
      mode="constant",
      cval=0.0,
      prefilter=False,
  )

  numerator = affine_transform(native_signal, **transform).astype(np.float32)
  support = affine_transform(native_validity, **transform).astype(np.float32)
  support = np.clip(support, 0.0, 1.0)
  canonical_volume = np.full(output_shape_zyx, np.nan, dtype=np.float32)
  supported = support > 1e-6
  canonical_volume[supported] = numerator[supported] / support[supported]

  metadata = {
      "input_shape_src": tuple(native_volume.shape),
      "output_shape_zyx": tuple(canonical_volume.shape),
      "output_spacing_xyz_mm": tuple(target_spacing_xyz_mm.tolist()),
      "output_origin_lps": tuple(lps_min.tolist()),
      "physical_extent_xyz_mm": tuple(physical_extent_xyz.tolist()),
      "canonical_affine_zyx_to_lps": canonical_affine,
  }
  return canonical_volume, support, metadata


def high_contrast_mr_window(
  volume,
  support,
  lower_percentile=20,
  upper_percentile=99,
  support_threshold=0.5,
):
  """Window supported voxels only; set unsupported output voxels to zero."""
  tensor = torch.as_tensor(volume, dtype=torch.float32)
  support_tensor = torch.as_tensor(support, dtype=torch.float32)
  if tensor.shape != support_tensor.shape:
      raise ValueError(f"Volume/support shape mismatch: {tensor.shape} vs {support_tensor.shape}")
  core = torch.isfinite(tensor) & (support_tensor >= support_threshold)
  finite_values = tensor[core]
  if finite_values.numel() == 0:
      raise ValueError("No supported voxels available for MRI intensity window")
  lower = torch.quantile(finite_values, lower_percentile / 100)
  upper = torch.quantile(finite_values, upper_percentile / 100)
  if upper <= lower:
      raise ValueError("Degenerate MRI intensity window")
  safe_tensor = torch.where(torch.isfinite(tensor), tensor, lower)
  windowed = ((safe_tensor - lower) / (upper - lower)).clamp(0, 1)
  has_support = torch.isfinite(tensor) & (support_tensor > 1e-6)
  windowed = torch.where(has_support, windowed, torch.zeros_like(windowed))
  return windowed, float(lower), float(upper)


def center_pad_or_crop_zyx(
  volume,
  target_shape_zyx,
):
  """
  Centre pad or crop a (z,y,x) tensor.

  Returns the processed volume and a record of each transformation.
  """
  output = torch.zeros(
      target_shape_zyx,
      dtype=volume.dtype,
  )

  source_slices = []
  destination_slices = []
  transformations = []

  for axis_name, current, target in zip(
      ("z", "y", "x"),
      volume.shape,
      target_shape_zyx,
  ):
      if current <= target:
          source_start = 0
          source_end = current

          destination_start = (
              target - current
          ) // 2
          destination_end = (
              destination_start + current
          )

          transformations.append(
              {
                  "axis": axis_name,
                  "operation": "pad",
                  "before": destination_start,
                  "after": (
                      target
                      - destination_end
                  ),
              }
          )
      else:
          source_start = (
              current - target
          ) // 2
          source_end = (
              source_start + target
          )

          destination_start = 0
          destination_end = target

          transformations.append(
              {
                  "axis": axis_name,
                  "operation": "crop",
                  "start": source_start,
                  "end": source_end,
              }
          )

      source_slices.append(
          slice(
              source_start,
              source_end,
          )
      )
      destination_slices.append(
          slice(
              destination_start,
              destination_end,
          )
      )

  output[
      tuple(destination_slices)
  ] = volume[
      tuple(source_slices)
  ]

  return output, transformations


canonical_volume, canonical_support, canonical_metadata = (
  canonical_resample_lps(
      volume,
      geometry["affine_array_to_lps"],
  )
)

windowed_volume, intensity_low, intensity_high = (
  high_contrast_mr_window(
      canonical_volume, canonical_support
  )
)

model_volume_zyx, spatial_transformations = (
  center_pad_or_crop_zyx(
      windowed_volume,
      TARGET_SHAPE_ZYX,
  )
)

# Model spatial convention is (H, W, D) = (y, x, z).
model_volume_hwd = (
  model_volume_zyx
  .permute(1, 2, 0)
  .contiguous()
)

# One knee series repeated into the checkpoint's three DCE channels.
model_input = (
  model_volume_hwd[
      None,
      None,
      ...,
  ]
  .expand(
      1,
      3,
      *model_volume_hwd.shape,
  )
)

print(
  "Native shape (slice,row,column):",
  volume.shape,
)
print(
  "Canonical shape (z,y,x):",
  canonical_volume.shape,
)
print(
  "Model input shape (B,C,H,W,D):",
  tuple(model_input.shape),
)
print(
  "Canonical physical extent (x,y,z mm):",
  tuple(
      round(value, 2)
      for value in canonical_metadata[
          "physical_extent_xyz_mm"
      ]
  ),
)
print(
  "High-contrast intensity window:",
  round(intensity_low, 3),
  round(intensity_high, 3),
)
print(
  "Spatial transformations:",
  spatial_transformations,
)
print(
  "Model value range:",
  float(model_input.min()),
  float(model_input.max()),
)
print(
  "Finite:",
  bool(
      torch.isfinite(model_input).all()
  ),
)
print(
  "Channels identical:",
  bool(
      torch.equal(
          model_input[:, 0],
          model_input[:, 1],
      )
      and torch.equal(
          model_input[:, 1],
          model_input[:, 2],
      )
  ),
)

assert model_input.shape == (
  1,
  3,
  384,
  384,
  192,
)

# Show the three canonical centre planes after preprocessing.
processed = model_volume_zyx.numpy()

fig, axes = plt.subplots(
  1,
  3,
  figsize=(14, 4),
)

axes[0].imshow(
  processed[
      processed.shape[0] // 2
  ],
  cmap="gray",
  vmin=0,
  vmax=1,
)
axes[0].set_title(
  "Canonical z centre"
)

axes[1].imshow(
  processed[
      :,
      processed.shape[1] // 2,
      :,
  ],
  cmap="gray",
  vmin=0,
  vmax=1,
  aspect=1,
)
axes[1].set_title(
  "Canonical y centre"
)

axes[2].imshow(
  processed[
      :,
      :,
      processed.shape[2] // 2,
  ],
  cmap="gray",
  vmin=0,
  vmax=1,
  aspect=1,
)
axes[2].set_title(
  "Canonical x centre"
)

for axis in axes:
  axis.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
### Cell 11 — generate the first Pillar-0 series embedding

import time

MODEL_INPUT_KEY = "breast_mr"
EXPECTED_EMBEDDING_DIM = 1152

if pillar.training:
  raise RuntimeError(
      "Pillar-0 must remain in evaluation mode."
  )

if any(
  parameter.requires_grad
  for parameter in pillar.parameters()
):
  raise RuntimeError(
      "Pillar-0 contains trainable parameters."
  )

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Materialize the expanded channel view and transfer it once.
gpu_input = (
  model_input
  .contiguous()
  .to(
      device=device,
      dtype=torch.float16,
      non_blocking=True,
  )
)

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
  with torch.autocast(
      device_type="cuda",
      dtype=torch.float16,
  ):
      first_output = (
          pillar.extract_vision_feats(
              {
                  MODEL_INPUT_KEY:
                      gpu_input
              }
          )
      )

torch.cuda.synchronize()
first_elapsed = (
  time.perf_counter() - start_time
)

if not isinstance(
  first_output,
  torch.Tensor,
):
  raise TypeError(
      "Expected Pillar-0 to return a tensor, "
      f"received {type(first_output)}"
  )

first_embedding = (
  first_output
  .squeeze(0)
  .float()
  .cpu()
)

if first_embedding.shape != (
  EXPECTED_EMBEDDING_DIM,
):
  raise ValueError(
      "Unexpected embedding shape: "
      f"{tuple(first_embedding.shape)}"
  )

if not torch.isfinite(
  first_embedding
).all():
  raise ValueError(
      "Embedding contains NaN or infinite values"
  )

# Repeat inference once to check deterministic eval behavior.
with torch.inference_mode():
  with torch.autocast(
      device_type="cuda",
      dtype=torch.float16,
  ):
      second_output = (
          pillar.extract_vision_feats(
              {
                  MODEL_INPUT_KEY:
                      gpu_input
              }
          )
      )

second_embedding = (
  second_output
  .squeeze(0)
  .float()
  .cpu()
)

maximum_repeat_difference = float(
  (
      first_embedding
      - second_embedding
  )
  .abs()
  .max()
)

embedding_norm = float(
  torch.linalg.vector_norm(
      first_embedding
  )
)

print(
  "Raw model output shape:",
  tuple(first_output.shape),
)
print(
  "Series embedding shape:",
  tuple(first_embedding.shape),
)
print(
  "Inference time (seconds):",
  round(first_elapsed, 3),
)
print(
  "Embedding minimum:",
  float(first_embedding.min()),
)
print(
  "Embedding maximum:",
  float(first_embedding.max()),
)
print(
  "Embedding mean:",
  float(first_embedding.mean()),
)
print(
  "Embedding standard deviation:",
  float(first_embedding.std()),
)
print(
  "Embedding L2 norm:",
  embedding_norm,
)
print(
  "Maximum repeated-inference difference:",
  maximum_repeat_difference,
)
print(
  "Peak GPU memory allocated (MB):",
  round(
      torch.cuda.max_memory_allocated()
      / 1024**2,
      1,
  ),
)
print(
  "Finite:",
  bool(
      torch.isfinite(
          first_embedding
      ).all()
  ),
)

series_embedding = first_embedding

del first_output
del second_output
del second_embedding
del gpu_input

torch.cuda.empty_cache()


In [ ]:
### Cell 12 — compare embeddings across the three planes

import gc
import torch.nn.functional as F

def preprocess_series_for_pillar(
  study_uid,
  series_uid,
):
  native_volume, native_geometry = (
      reconstruct_series(
          study_uid,
          series_uid,
      )
  )

  canonical, support_canonical, canonical_info = (
      canonical_resample_lps(
          native_volume,
          native_geometry[
              "affine_array_to_lps"
          ],
      )
  )

  windowed, window_low, window_high = (
      high_contrast_mr_window(
          canonical, support_canonical
      )
  )

  padded, spatial_operations = (
      center_pad_or_crop_zyx(
          windowed,
          TARGET_SHAPE_ZYX,
      )
  )

  # Canonical (z,y,x) -> model (H,W,D) = (y,x,z).
  model_hwd = (
      padded
      .permute(1, 2, 0)
      .contiguous()
  )

  model_tensor = (
      model_hwd[
          None,
          None,
          ...,
      ]
      .expand(
          1,
          3,
          *model_hwd.shape,
      )
  )

  preprocessing_record = {
      "native_shape_drc":
          tuple(native_volume.shape),
      "native_finite_voxel_fraction": float(np.isfinite(native_volume).mean()),
      "native_nan_or_infinite_voxel_fraction": float(1.0 - np.isfinite(native_volume).mean()),
      "native_spacing_drc_mm":
          tuple(
              native_geometry[
                  "spacing_drc_mm"
              ]
          ),
      "canonical_shape_zyx":
          tuple(canonical.shape),
      "canonical_extent_xyz_mm":
          tuple(
              canonical_info[
                  "physical_extent_xyz_mm"
              ]
          ),
      "model_shape_bchwd":
          tuple(model_tensor.shape),
      "window_low":
          window_low,
      "window_high":
          window_high,
      "canonical_support_fraction": float(np.mean(support_canonical)),
      "intensity_window_support_threshold": 0.5,
      "spatial_operations":
          spatial_operations,
      "coordinate_system":
          "canonical DICOM LPS",
      "channel_adapter":
          "repeat single series 3 times",
  }

  return (
      model_tensor,
      preprocessing_record,
  )


@torch.inference_mode()
def extract_pillar_embedding(
  model_tensor,
):
  gpu_tensor = (
      model_tensor
      .contiguous()
      .to(
          device=device,
          dtype=torch.float16,
          non_blocking=True,
      )
  )

  with torch.autocast(
      device_type="cuda",
      dtype=torch.float16,
  ):
      output = (
          pillar.extract_vision_feats(
              {
                  "breast_mr":
                      gpu_tensor
              }
          )
      )

  embedding = (
      output
      .squeeze(0)
      .float()
      .cpu()
  )

  if embedding.shape != (1152,):
      raise ValueError(
          "Unexpected embedding shape: "
          f"{tuple(embedding.shape)}"
      )

  if not torch.isfinite(
      embedding
  ).all():
      raise ValueError(
          "Embedding contains non-finite values"
      )

  del gpu_tensor
  del output

  torch.cuda.empty_cache()

  return embedding


# Release the large preprocessing tensors from Cell 10.
for variable_name in [
  "model_input",
  "model_volume_hwd",
  "model_volume_zyx",
  "windowed_volume",
  "canonical_volume",
]:
  globals().pop(
      variable_name,
      None,
  )

gc.collect()
torch.cuda.empty_cache()

plane_embeddings = []
plane_records = []

for row in representative_rows.itertuples(
  index=False
):
  print(
      "Encoding",
      row.Anatomical_Plane,
      row.SeriesInstanceUID,
  )

  prepared_tensor, preprocessing_record = (
      preprocess_series_for_pillar(
          row.StudyInstanceUID,
          row.SeriesInstanceUID,
      )
  )

  embedding = extract_pillar_embedding(
      prepared_tensor
  )

  plane_embeddings.append(
      embedding
  )

  plane_records.append(
      {
          "plane": row.Anatomical_Plane,
          "fluid_sensitive":
              row.Fluid_Sensitive,
          "fat_suppression":
              row.Fat_Suppression,
          "embedding_norm": float(
              torch.linalg.vector_norm(
                  embedding
              )
          ),
          "native_shape":
              preprocessing_record[
                  "native_shape_drc"
              ],
          "canonical_shape":
              preprocessing_record[
                  "canonical_shape_zyx"
              ],
          "window": (
              round(
                  preprocessing_record[
                      "window_low"
                  ],
                  3,
              ),
              round(
                  preprocessing_record[
                      "window_high"
                  ],
                  3,
              ),
          ),
      }
  )

  del prepared_tensor
  gc.collect()

embedding_matrix = torch.stack(
  plane_embeddings
)

normalized_embeddings = F.normalize(
  embedding_matrix,
  dim=1,
)

cosine_similarity = (
  normalized_embeddings
  @ normalized_embeddings.T
)

plane_names = [
  record["plane"]
  for record in plane_records
]

similarity_table = pd.DataFrame(
  cosine_similarity.numpy(),
  index=plane_names,
  columns=plane_names,
)

display(
  pd.DataFrame(plane_records)
)

print("Cosine similarity:")
display(
  similarity_table.style.format(
      "{:.4f}"
  )
)

off_diagonal_mask = ~torch.eye(
  len(plane_embeddings),
  dtype=torch.bool,
)

off_diagonal_values = (
  cosine_similarity[
      off_diagonal_mask
  ]
)

print(
  "Off-diagonal similarity range:",
  round(
      float(
          off_diagonal_values.min()
      ),
      4,
  ),
  "to",
  round(
      float(
          off_diagonal_values.max()
      ),
      4,
  ),
)


In [ ]:
  ### Cell 13 — create the small series-embedding cache

  import hashlib
  import json
  import time
  from pathlib import Path


  MAX_SANITY_STUDIES = 8

  gold = (
      train.loc[
          studies_with_all_labels,
          ["StudyInstanceUID", *TARGETS],
      ]
      .copy()
  )

  gold["StudyInstanceUID"] = (
      gold["StudyInstanceUID"].astype(str)
  )

  # Select a small set that covers both binary values
  # for every target.
  required_states = {
      (target, value)
      for target in TARGETS
      for value in (0, 1)
  }

  study_states = {}

  for row in gold.itertuples(index=False):
      study_uid = str(row.StudyInstanceUID)

      study_states[study_uid] = {
          (
              target,
              int(
                  gold.loc[
                      gold["StudyInstanceUID"]
                      == study_uid,
                      target,
                  ].iloc[0]
              ),
          )
          for target in TARGETS
      }

  selected_studies = []
  covered_states = set()
  remaining_studies = sorted(
      study_states
  )

  while (
      covered_states != required_states
      and len(selected_studies)
      < MAX_SANITY_STUDIES
  ):
      best_study = max(
          remaining_studies,
          key=lambda study_uid: (
              len(
                  study_states[study_uid]
                  - covered_states
              ),
              study_uid,
          ),
      )

      new_states = (
          study_states[best_study]
          - covered_states
      )

      if not new_states:
          break

      selected_studies.append(
          best_study
      )
      covered_states.update(
          study_states[best_study]
      )
      remaining_studies.remove(
          best_study
      )

  missing_states = (
      required_states - covered_states
  )

  if missing_states:
      raise RuntimeError(
          "The small sample does not cover all "
          f"target states: {sorted(missing_states)}"
      )

  selected_labels = (
      gold[
          gold["StudyInstanceUID"]
          .isin(selected_studies)
      ]
      .set_index("StudyInstanceUID")
      .loc[selected_studies]
  )

  selected_series = (
      train_series[
          train_series["StudyInstanceUID"]
          .isin(selected_studies)
      ]
      .sort_values(
          [
              "StudyInstanceUID",
              "Anatomical_Plane",
              "SeriesInstanceUID",
          ]
      )
      .reset_index(drop=True)
  )

  print(
      "Selected studies:",
      len(selected_studies),
  )
  print(
      "Selected series:",
      len(selected_series),
  )
  print(
      "Series per selected study:"
  )
  display(
      selected_series
      .groupby("StudyInstanceUID")
      .size()
      .rename("series")
      .to_frame()
  )

  print(
      "Label coverage in selected studies:"
  )
  display(
      pd.DataFrame(
          {
              "positive":
                  selected_labels[
                      TARGETS
                  ].eq(1).sum(),
              "negative":
                  selected_labels[
                      TARGETS
                  ].eq(0).sum(),
          }
      )
  )

  cache_config = {
      "checkpoint": MODEL_ID,
      "checkpoint_revision":
          PILLAR_REVISION,
      "embedding_dim": 1152,
      "model_input_key": "breast_mr",
      "target_spacing_xyz_mm":
          [1.0, 1.0, 1.0],
      "target_shape_zyx":
          list(TARGET_SHAPE_ZYX),
      "intensity_window":
          "MR percentiles 20-99 over resampled voxels with source support >= 0.5",
      "preprocessing_revision": "mask_aware_resampling_v1",
      "coordinate_system":
          "canonical DICOM LPS",
      "channel_adapter":
          "repeat one knee series into 3 channels",
      "selection":
          "greedy binary-target coverage",
      "selection_limit":
          MAX_SANITY_STUDIES,
  }

  cache_config_json = json.dumps(
      cache_config,
      sort_keys=True,
  )

  cache_fingerprint = (
      hashlib.sha256(
          cache_config_json.encode("utf-8")
      )
      .hexdigest()[:16]
  )

  CACHE_ROOT = (
      Path("/kaggle/working")
      / "pillar0_knee_cache"
      / cache_fingerprint
  )

  CACHE_ROOT.mkdir(
      parents=True,
      exist_ok=True,
  )

  (CACHE_ROOT / "config.json").write_text(
      json.dumps(
          cache_config,
          indent=2,
      )
  )

  selected_labels.to_csv(
      CACHE_ROOT / "selected_labels.csv"
  )

  manifest_rows = []
  cache_start = time.perf_counter()

  for series_index, row in enumerate(
      selected_series.itertuples(index=False),
      start=1,
  ):
      study_uid = str(
          row.StudyInstanceUID
      )
      series_uid = str(
          row.SeriesInstanceUID
      )

      cache_path = (
          CACHE_ROOT
          / f"{series_uid}.pt"
      )

      if cache_path.exists():
          cached = torch.load(
              cache_path,
              map_location="cpu",
              weights_only=False,
          )

          if (
              cached["checkpoint_revision"]
              != PILLAR_REVISION
              or cached["cache_fingerprint"]
              != cache_fingerprint
          ):
              raise RuntimeError(
                  "Existing cache provenance mismatch"
              )

          embedding = cached["embedding"]
          cache_status = "reused"

      else:
          prepared_tensor, preprocessing_record = (
              preprocess_series_for_pillar(
                  study_uid,
                  series_uid,
              )
          )

          embedding = (
              extract_pillar_embedding(
                  prepared_tensor
              )
          )

          cached = {
              "study_uid":
                  study_uid,
              "series_uid":
                  series_uid,
              "embedding":
                  embedding,
              "anatomical_plane":
                  str(row.Anatomical_Plane),
              "fluid_sensitive":
                  int(row.Fluid_Sensitive),
              "fat_suppression":
                  int(row.Fat_Suppression),
              "preprocessing":
                  preprocessing_record,
              "checkpoint":
                  MODEL_ID,
              "checkpoint_revision":
                  PILLAR_REVISION,
              "cache_fingerprint":
                  cache_fingerprint,
          }

          torch.save(
              cached,
              cache_path,
          )

          del prepared_tensor
          cache_status = "created"

      if embedding.shape != (1152,):
          raise ValueError(
              f"Bad cached embedding shape: "
              f"{embedding.shape}"
          )

      if not torch.isfinite(
          embedding
      ).all():
          raise ValueError(
              "Cached embedding is not finite"
          )

      manifest_rows.append(
          {
              "study_uid": study_uid,
              "series_uid": series_uid,
              "anatomical_plane":
                  str(row.Anatomical_Plane),
              "fluid_sensitive":
                  int(row.Fluid_Sensitive),
              "fat_suppression":
                  int(row.Fat_Suppression),
              "embedding_dim":
                  embedding.numel(),
              "embedding_norm":
                  float(
                      torch.linalg.vector_norm(
                          embedding
                      )
                  ),
              "cache_status":
                  cache_status,
              "cache_file":
                  cache_path.name,
          }
      )

      print(
          f"{series_index:02d}/"
          f"{len(selected_series):02d}",
          row.Anatomical_Plane,
          cache_status,
      )

  manifest = pd.DataFrame(
      manifest_rows
  )

  manifest.to_csv(
      CACHE_ROOT / "manifest.csv",
      index=False,
  )

  elapsed = (
      time.perf_counter()
      - cache_start
  )

  print()
  print("Cache directory:", CACHE_ROOT)
  print("Cached series:", len(manifest))
  print(
      "Cached studies:",
      manifest["study_uid"].nunique(),
  )
  print(
      "Embedding dimensions:",
      manifest["embedding_dim"].unique(),
  )
  print(
      "Embedding norm range:",
      manifest["embedding_norm"].min(),
      "to",
      manifest["embedding_norm"].max(),
  )
  print(
      "Elapsed seconds:",
      round(elapsed, 1),
  )

  display(manifest)



In [ ]:
### Cell 14 — assemble the variable-series study batch

import torch.nn as nn
import torch.nn.functional as F


PLANE_TO_INDEX = {
  "Axial": 0,
  "Coronal": 1,
  "Sagittal": 2,
}

PLANE_NAMES = [
  "Axial",
  "Coronal",
  "Sagittal",
]


def make_series_metadata_vector(
  cached_record,
):
  """
  Metadata vector:
    3 plane indicators
    1 fluid-sensitive indicator
    1 fat-suppression indicator
    3 native voxel spacings
    1 native slice count
  """
  plane_vector = torch.zeros(
      len(PLANE_NAMES),
      dtype=torch.float32,
  )

  plane = cached_record[
      "anatomical_plane"
  ]

  if plane not in PLANE_TO_INDEX:
      raise ValueError(
          f"Unknown anatomical plane: {plane}"
      )

  plane_vector[
      PLANE_TO_INDEX[plane]
  ] = 1.0

  preprocessing = cached_record[
      "preprocessing"
  ]

  native_spacing = torch.tensor(
      preprocessing[
          "native_spacing_drc_mm"
      ],
      dtype=torch.float32,
  )

  native_slice_count = float(
      preprocessing[
          "native_shape_drc"
      ][0]
  )

  return torch.cat(
      [
          plane_vector,
          torch.tensor(
              [
                  float(
                      cached_record[
                          "fluid_sensitive"
                      ]
                  ),
                  float(
                      cached_record[
                          "fat_suppression"
                      ]
                  ),
              ],
              dtype=torch.float32,
          ),
          native_spacing,
          torch.tensor(
              [native_slice_count],
              dtype=torch.float32,
          ),
      ]
  )


cache_manifest = pd.read_csv(
  CACHE_ROOT / "manifest.csv",
  dtype={
      "study_uid": str,
      "series_uid": str,
  },
)

cached_labels = pd.read_csv(
  CACHE_ROOT / "selected_labels.csv",
  dtype={
      "StudyInstanceUID": str,
  },
).set_index("StudyInstanceUID")

study_ids = sorted(
  cache_manifest[
      "study_uid"
  ].unique()
)

records_by_study = {
  study_uid: []
  for study_uid in study_ids
}

for row in cache_manifest.itertuples(
  index=False
):
  cache_path = (
      CACHE_ROOT
      / row.cache_file
  )

  record = torch.load(
      cache_path,
      map_location="cpu",
      weights_only=False,
  )

  if (
      record["study_uid"]
      != row.study_uid
  ):
      raise ValueError(
          "Cache study UID mismatch"
      )

  if (
      record["series_uid"]
      != row.series_uid
  ):
      raise ValueError(
          "Cache series UID mismatch"
      )

  if record["embedding"].shape != (
      1152,
  ):
      raise ValueError(
          "Cache embedding shape mismatch"
      )

  records_by_study[
      row.study_uid
  ].append(record)

# Use a deterministic series order for storage and debugging.
# The model itself will receive no series-position encoding.
for study_uid in study_ids:
  records_by_study[
      study_uid
  ].sort(
      key=lambda record: (
          record["anatomical_plane"],
          record["series_uid"],
      )
  )

batch_size = len(study_ids)
maximum_series = max(
  len(records_by_study[study_uid])
  for study_uid in study_ids
)

embedding_dimension = 1152
metadata_dimension = 9
number_of_targets = len(TARGETS)

series_embeddings = torch.zeros(
  batch_size,
  maximum_series,
  embedding_dimension,
  dtype=torch.float32,
)

series_metadata = torch.zeros(
  batch_size,
  maximum_series,
  metadata_dimension,
  dtype=torch.float32,
)

# True means that a series position is padding.
series_padding_mask = torch.ones(
  batch_size,
  maximum_series,
  dtype=torch.bool,
)

study_labels = torch.zeros(
  batch_size,
  number_of_targets,
  dtype=torch.float32,
)

for study_index, study_uid in enumerate(
  study_ids
):
  study_records = records_by_study[
      study_uid
  ]

  for series_index, record in enumerate(
      study_records
  ):
      series_embeddings[
          study_index,
          series_index,
      ] = record["embedding"]

      series_metadata[
          study_index,
          series_index,
      ] = make_series_metadata_vector(
          record
      )

      series_padding_mask[
          study_index,
          series_index,
      ] = False

  study_labels[
      study_index
  ] = torch.tensor(
      cached_labels.loc[
          study_uid,
          TARGETS,
      ].to_numpy(
          dtype=np.float32
      )
  )

# Normalize only the continuous metadata:
# spacing_d, spacing_r, spacing_c, and slice count.
valid_series_mask = (
  ~series_padding_mask
)

continuous_metadata = (
  series_metadata[:, :, 5:]
)

valid_continuous_metadata = (
  continuous_metadata[
      valid_series_mask
  ]
)

metadata_mean = (
  valid_continuous_metadata.mean(
      dim=0
  )
)

metadata_std = (
  valid_continuous_metadata.std(
      dim=0
  )
  .clamp_min(1e-6)
)

continuous_metadata[
  valid_series_mask
] = (
  valid_continuous_metadata
  - metadata_mean
) / metadata_std

# Keep padded metadata exactly zero.
series_metadata[
  series_padding_mask
] = 0

series_counts = (
  ~series_padding_mask
).sum(dim=1)

print(
  "Study IDs:",
  len(study_ids),
)
print(
  "Series counts:",
  series_counts.tolist(),
)
print(
  "Series embedding batch:",
  tuple(series_embeddings.shape),
)
print(
  "Series metadata batch:",
  tuple(series_metadata.shape),
)
print(
  "Padding mask:",
  tuple(series_padding_mask.shape),
)
print(
  "Study labels:",
  tuple(study_labels.shape),
)
print(
  "Metadata mean before normalization:",
  metadata_mean.tolist(),
)
print(
  "Metadata std before normalization:",
  metadata_std.tolist(),
)
print(
  "All embeddings finite:",
  bool(
      torch.isfinite(
          series_embeddings
      ).all()
  ),
)
print(
  "All metadata finite:",
  bool(
      torch.isfinite(
          series_metadata
      ).all()
  ),
)
print(
  "All labels binary:",
  bool(
      torch.all(
          (study_labels == 0)
          | (study_labels == 1)
      )
  ),
)

batch_summary = pd.DataFrame(
  {
      "study_uid": study_ids,
      "series_count":
          series_counts.tolist(),
      "padding_count": (
          maximum_series
          - series_counts
      ).tolist(),
      "positive_labels":
          study_labels.sum(
              dim=1
          ).tolist(),
  }
)

display(batch_summary)


In [ ]:
### Cell 15 — define and verify the study aggregator

class StudyAggregator(nn.Module):
  def __init__(
      self,
      series_embedding_dim=1152,
      series_metadata_dim=9,
      hidden_dim=128,
      attention_heads=4,
      number_of_targets=12,
  ):
      super().__init__()

      self.series_projection = nn.Linear(
          series_embedding_dim,
          hidden_dim,
      )

      self.metadata_projection = nn.Linear(
          series_metadata_dim,
          hidden_dim,
      )

      # This token gathers information from all real series.
      self.study_token = nn.Parameter(
          torch.zeros(
              1,
              1,
              hidden_dim,
          )
      )

      encoder_layer = (
          nn.TransformerEncoderLayer(
              d_model=hidden_dim,
              nhead=attention_heads,
              dim_feedforward=(
                  hidden_dim * 2
              ),
              dropout=0.0,
              activation="gelu",
              batch_first=True,
              norm_first=True,
          )
      )

      self.encoder = nn.TransformerEncoder(
          encoder_layer,
          num_layers=1,
          enable_nested_tensor=False,
      )

      self.study_norm = nn.LayerNorm(
          hidden_dim
      )

      self.classifier = nn.Linear(
          hidden_dim,
          number_of_targets,
      )

      nn.init.normal_(
          self.study_token,
          mean=0.0,
          std=0.02,
      )

  def forward(
      self,
      embeddings,
      metadata,
      padding_mask,
  ):
      if embeddings.ndim != 3:
          raise ValueError(
              "Expected embeddings shaped "
              "[batch, series, features]"
          )

      if metadata.shape[:2] != (
          embeddings.shape[:2]
      ):
          raise ValueError(
              "Embedding and metadata dimensions "
              "do not match"
          )

      if padding_mask.shape != (
          embeddings.shape[:2]
      ):
          raise ValueError(
              "Padding mask dimensions do not match"
          )

      series_tokens = (
          self.series_projection(
              embeddings
          )
          + self.metadata_projection(
              metadata
          )
      )

      study_token = (
          self.study_token.expand(
              embeddings.shape[0],
              -1,
              -1,
          )
      )

      tokens = torch.cat(
          [
              study_token,
              series_tokens,
          ],
          dim=1,
      )

      # The study token is always a real token.
      study_token_mask = torch.zeros(
          (
              padding_mask.shape[0],
              1,
          ),
          dtype=torch.bool,
          device=padding_mask.device,
      )

      transformer_mask = torch.cat(
          [
              study_token_mask,
              padding_mask,
          ],
          dim=1,
      )

      encoded = self.encoder(
          tokens,
          src_key_padding_mask=(
              transformer_mask
          ),
      )

      z_study = self.study_norm(
          encoded[:, 0]
      )

      logits = self.classifier(
          z_study
      )

      return logits, z_study


torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

aggregator = StudyAggregator(
  series_embedding_dim=1152,
  series_metadata_dim=9,
  hidden_dim=128,
  attention_heads=4,
  number_of_targets=len(TARGETS),
).to(device)

aggregator_parameter_count = sum(
  parameter.numel()
  for parameter in aggregator.parameters()
  if parameter.requires_grad
)

batch_embeddings_gpu = (
  series_embeddings.to(device)
)

batch_metadata_gpu = (
  series_metadata.to(device)
)

batch_padding_mask_gpu = (
  series_padding_mask.to(device)
)

aggregator.eval()

with torch.inference_mode():
  initial_logits, initial_z_study = (
      aggregator(
          batch_embeddings_gpu,
          batch_metadata_gpu,
          batch_padding_mask_gpu,
      )
  )

print(
  "Trainable aggregator parameters:",
  f"{aggregator_parameter_count:,}",
)
print(
  "Logit shape:",
  tuple(initial_logits.shape),
)
print(
  "Study embedding shape:",
  tuple(initial_z_study.shape),
)
print(
  "Finite logits:",
  bool(
      torch.isfinite(
          initial_logits
      ).all()
  ),
)
print(
  "Finite study embeddings:",
  bool(
      torch.isfinite(
          initial_z_study
      ).all()
  ),
)

# Verify that series ordering does not change z_study.
series_permutation = torch.tensor(
  list(
      reversed(
          range(maximum_series)
      )
  ),
  device=device,
)

with torch.inference_mode():
  permuted_logits, permuted_z_study = (
      aggregator(
          batch_embeddings_gpu[
              :,
              series_permutation,
          ],
          batch_metadata_gpu[
              :,
              series_permutation,
          ],
          batch_padding_mask_gpu[
              :,
              series_permutation,
          ],
      )
  )

maximum_embedding_difference = float(
  (
      initial_z_study
      - permuted_z_study
  )
  .abs()
  .max()
)

maximum_logit_difference = float(
  (
      initial_logits
      - permuted_logits
  )
  .abs()
  .max()
)

print(
  "Maximum z_study difference after "
  "series permutation:",
  maximum_embedding_difference,
)
print(
  "Maximum logit difference after "
  "series permutation:",
  maximum_logit_difference,
)

assert initial_logits.shape == (
  len(study_ids),
  len(TARGETS),
)

assert initial_z_study.shape == (
  len(study_ids),
  128,
)

assert maximum_embedding_difference < 1e-5
assert maximum_logit_difference < 1e-5


In [ ]:
### Cell 16 — tiny overfit, export probabilities and z_study

import matplotlib.pyplot as plt


TRAINING_STEPS = 300
LEARNING_RATE = 3e-3
WEIGHT_DECAY = 1e-4

# Recreate the model so this cell is reproducible when rerun.
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

aggregator = StudyAggregator(
  series_embedding_dim=1152,
  series_metadata_dim=9,
  hidden_dim=128,
  attention_heads=4,
  number_of_targets=len(TARGETS),
).to(device)

optimizer = torch.optim.AdamW(
  aggregator.parameters(),
  lr=LEARNING_RATE,
  weight_decay=WEIGHT_DECAY,
)

loss_function = nn.BCEWithLogitsLoss()

labels_gpu = study_labels.to(device)

loss_history = []

aggregator.train()

for step in range(
  TRAINING_STEPS + 1
):
  logits, z_study = aggregator(
      batch_embeddings_gpu,
      batch_metadata_gpu,
      batch_padding_mask_gpu,
  )

  loss = loss_function(
      logits,
      labels_gpu,
  )

  loss_history.append(
      float(loss.detach().cpu())
  )

  if step % 50 == 0:
      print(
          f"step {step:03d} | "
          f"loss {loss_history[-1]:.6f}"
      )

  if step == TRAINING_STEPS:
      break

  optimizer.zero_grad(
      set_to_none=True
  )

  loss.backward()

  torch.nn.utils.clip_grad_norm_(
      aggregator.parameters(),
      max_norm=1.0,
  )

  optimizer.step()

initial_loss = loss_history[0]
final_training_loss = loss_history[-1]

if not (
  final_training_loss
  < initial_loss
):
  raise RuntimeError(
      "Sanity-check loss did not decrease"
  )

aggregator.eval()

with torch.inference_mode():
  final_logits, final_z_study = (
      aggregator(
          batch_embeddings_gpu,
          batch_metadata_gpu,
          batch_padding_mask_gpu,
      )
  )

  final_probabilities = (
      torch.sigmoid(
          final_logits
      )
  )

final_loss = float(
  loss_function(
      final_logits,
      labels_gpu,
  ).cpu()
)

if not torch.isfinite(
  final_z_study
).all():
  raise ValueError(
      "z_study contains non-finite values"
  )

if not torch.isfinite(
  final_probabilities
).all():
  raise ValueError(
      "Probabilities contain non-finite values"
  )

print()
print(
  "Initial loss:",
  round(initial_loss, 6),
)
print(
  "Final loss:",
  round(final_loss, 6),
)
print(
  "Logit shape:",
  tuple(final_logits.shape),
)
print(
  "Probability shape:",
  tuple(
      final_probabilities.shape
  ),
)
print(
  "z_study shape:",
  tuple(final_z_study.shape),
)
print(
  "Probability range:",
  float(
      final_probabilities.min()
  ),
  "to",
  float(
      final_probabilities.max()
  ),
)

# Plot optimization behavior.
fig, axis = plt.subplots(
  figsize=(7, 4)
)

axis.plot(
  loss_history,
  linewidth=2,
)

axis.set_xlabel("Training step")
axis.set_ylabel(
  "BCEWithLogitsLoss"
)
axis.set_title(
  "Four-study overfit sanity check"
)
axis.grid(
  alpha=0.25
)

plt.tight_layout()
plt.show()

# Keep continuous probabilities; do not threshold them.
probabilities_cpu = (
  final_probabilities
  .float()
  .cpu()
)

logits_cpu = (
  final_logits
  .float()
  .cpu()
)

z_study_cpu = (
  final_z_study
  .float()
  .cpu()
)

prediction_table = pd.DataFrame(
  {
      "StudyInstanceUID":
          study_ids
  }
)

for target_index, target in enumerate(
  TARGETS
):
  prediction_table[
      f"label__{target}"
  ] = (
      study_labels[
          :,
          target_index,
      ].numpy()
  )

  prediction_table[
      f"probability__{target}"
  ] = (
      probabilities_cpu[
          :,
          target_index,
      ].numpy()
  )

display(prediction_table)

# Inspect study-space similarity without altering exported z_study.
normalized_z = F.normalize(
  z_study_cpu,
  dim=1,
)

study_similarity = (
  normalized_z
  @ normalized_z.T
)

display(
  pd.DataFrame(
      study_similarity.numpy(),
      index=study_ids,
      columns=study_ids,
  ).style.format("{:.4f}")
)

EXPERIMENT_DIR = (
  CACHE_ROOT
  / "aggregator_sanity"
)

EXPERIMENT_DIR.mkdir(
  parents=True,
  exist_ok=True,
)

training_config = {
  "purpose":
      "end-to-end overfit sanity check",
  "seed":
      SEED,
  "training_studies":
      len(study_ids),
  "training_series":
      int(
          series_counts.sum()
      ),
  "targets":
      TARGETS,
  "loss":
      "BCEWithLogitsLoss",
  "class_weighting":
      None,
  "learning_rate":
      LEARNING_RATE,
  "weight_decay":
      WEIGHT_DECAY,
  "steps":
      TRAINING_STEPS,
  "aggregator_hidden_dim":
      128,
  "attention_heads":
      4,
  "transformer_layers":
      1,
  "study_embedding_dim":
      128,
  "series_position_encoding":
      None,
  "cache_fingerprint":
      cache_fingerprint,
  "pillar_checkpoint":
      MODEL_ID,
  "pillar_revision":
      PILLAR_REVISION,
  "initial_loss":
      initial_loss,
  "final_loss":
      final_loss,
}

with (
  EXPERIMENT_DIR
  / "training_config.json"
).open("w") as file:
  json.dump(
      training_config,
      file,
      indent=2,
  )

torch.save(
  {
      "model_state_dict":
          aggregator.state_dict(),
      "metadata_mean":
          metadata_mean,
      "metadata_std":
          metadata_std,
      "training_config":
          training_config,
  },
  EXPERIMENT_DIR
  / "aggregator.pt",
)

pd.DataFrame(
  {
      "step": range(
          len(loss_history)
      ),
      "loss": loss_history,
  }
).to_csv(
  EXPERIMENT_DIR
  / "loss_history.csv",
  index=False,
)

prediction_table.to_csv(
  EXPERIMENT_DIR
  / "study_predictions.csv",
  index=False,
)

np.savez_compressed(
  EXPERIMENT_DIR
  / "study_representations.npz",
  study_uid=np.asarray(
      study_ids
  ),
  z_study=z_study_cpu.numpy(),
  logits=logits_cpu.numpy(),
  probabilities=(
      probabilities_cpu.numpy()
  ),
  labels=study_labels.numpy(),
  target_names=np.asarray(
      TARGETS
  ),
)

print()
print(
  "Experiment artifacts:",
  EXPERIMENT_DIR,
)
print(
  "Saved files:",
  sorted(
      path.name
      for path
      in EXPERIMENT_DIR.iterdir()
  ),
)
